# components.page_renderer

> Full session management page assembly — header with title + cross-management tabs, and the session list body.

The page header uses `render_management_tabs` (from `components.helpers`) to present Documents / Sessions as cross-page navigation. The `active` tab is always "sessions" on this page; the tab entries are supplied by the host via the `tab_entries` argument so the host decides which sibling pages to link to.

In [ ]:
#| default_exp components.page_renderer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import Div, H1

from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import container, max_w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap,
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_fasthtml_workflow_session_management.models import SessionManagementUrls
from cjm_fasthtml_workflow_session_management.html_ids import SessionManagerHtmlIds
from cjm_fasthtml_workflow_session_management.components.helpers import (
    render_management_tabs, DEBUG_SESSION_RENDER,
)

In [ ]:
#| export
def render_page_header(
    title:str="Sessions", # Page title
    icon_name:str="layers", # Lucide icon for the title
    tab_entries:Optional[List[Tuple[str, str, str, str]]]=None, # Optional (key, label, icon, url) for cross-mgmt tabs
    active_tab:str="sessions", # Key of the currently active tab
) -> Any: # Header element
    """Render the page header with title and optional cross-management tab navigation."""
    children = [
        H1(
            lucide_icon(icon_name, size=7),
            title,
            cls=combine_classes(
                font_size._3xl, font_weight.bold,
                flex_display, items.center, gap(3),
            ),
        ),
    ]
    if tab_entries:
        children.append(render_management_tabs(active=active_tab, entries=tab_entries))
    return Div(
        *children,
        cls=combine_classes(
            flex_display, flex_direction.col, gap(3), m.b(4),
        ),
    )

In [ ]:
# Header without tabs.
h = render_page_header()
assert h.tag == "div"
assert len(h.children) == 1  # just the title
assert h.children[0].tag == "h1"

# Header with tabs.
entries = [
    ("documents", "Documents", "file-text", "/manage/documents"),
    ("sessions", "Sessions", "layers", "/manage/sessions"),
]
h2 = render_page_header(title="Manage", tab_entries=entries, active_tab="sessions")
assert len(h2.children) == 2  # title + tabs
tabs_row = h2.children[1]
assert tabs_row.attrs["role"] == "tablist"
# Sessions should be the active tab.
sessions_tab = tabs_row.children[1]
assert "tab-active" in sessions_tab.attrs["class"]
print("Page header OK")

Page header OK


## Full Page

`render_session_manager_page` takes a late-bound `render_list_fn` so state mutations are reflected at render time (not at closure-capture time).

In [ ]:
#| export
def render_session_manager_page(
    urls:SessionManagementUrls, # URL bundle (unused directly, reserved for future header actions)
    render_list_fn:Callable, # () -> session list component
    title:str="Sessions", # Page title
    icon_name:str="layers", # Lucide icon name for the title
    tab_entries:Optional[List[Tuple[str, str, str, str]]]=None, # Cross-management tabs
    active_tab:str="sessions", # Currently active tab key
) -> Any: # Complete session manager page component
    """Render the complete session manager page with header and session list."""
    if DEBUG_SESSION_RENDER:
        print(f"[RENDER] session_manager_page")
    return Div(
        render_page_header(
            title=title,
            icon_name=icon_name,
            tab_entries=tab_entries,
            active_tab=active_tab,
        ),
        render_list_fn(),
        id=SessionManagerHtmlIds.PAGE_CONTENT,
        cls=combine_classes(
            container, max_w._5xl, m.x.auto,
            flex_display, flex_direction.col,
            p(4), gap(4), h.full,
        ),
    )

In [ ]:
# Late-bound render_list_fn is called at render time, not capture time.
_call_count = [0]
def _list_stub():
    _call_count[0] += 1
    return Div("list content", id="stub-list")

_urls = SessionManagementUrls(
    management_page="/m/sessions/", list_sessions="/m/sessions/list",
    session_detail="/m/sessions/detail", create_session="/m/sessions/create",
    delete_session="/m/sessions/delete", rename_session="/m/sessions/rename",
    resume_session="/m/sessions/resume",
)
page = render_session_manager_page(_urls, _list_stub)
assert _call_count[0] == 1, "render_list_fn should be called exactly once per render"
assert page.tag == "div"
assert page.attrs["id"] == SessionManagerHtmlIds.PAGE_CONTENT
assert page.children[1].attrs["id"] == "stub-list"
print(f"Page OK: {len(page.children)} children")

Page OK: 2 children


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()